# Simple Plutus Learning Recommendation System

Built on the **Plutus Learner RAG** corpus.

- **Model**: Content-based recommender using pure-Python BM25 (or optional FastEmbed hybrid)
- **Training**: Instant index loading / optional embedding build on CPU
- **Deployment**: Hugging Face Gradio CPU Space

Upload the original `plutus_rag_bundle.zip` when prompted.

In [ ]:
from pathlib import Path
import zipfile, sys
from google.colab import files

print("Upload plutus_rag_bundle.zip")
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith(".zip"))

ROOT = Path("/content/plutus_recommender")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

# Move the extracted plutus_rag into our working dir if needed
extracted = Path("/content/plutus_rag")
if extracted.exists():
    import shutil
    if ROOT.exists():
        shutil.rmtree(ROOT)
    extracted.rename(ROOT)
print("Ready:", ROOT)
print(list(ROOT.glob("*.py")))

## 1. Load the recommender ("train" = load index)

In [ ]:
sys.path.insert(0, str(ROOT))
from recommender import PlutusRecommender

# If you want hybrid dense mode, first run:
# !pip install -q fastembed numpy
# from rag import embed_index
# embed_index(ROOT / "data/index")
# rec = PlutusRecommender(ROOT / "data/index", mode="hybrid")

rec = PlutusRecommender(ROOT / "data/index", mode="bm25")
print("Topics available:")
for t in rec.list_topics():
    print(" -", t)

## 2. Get recommendations

In [ ]:
query = "difference between datum and redeemer in Plutus"
results = rec.recommend(query, k=5)

print(f"Recommendations for: {query}\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['score']}] {r['title']}")
    print(f"   {r['url']}")
    print(f"   {r['snippet'][:180]}...\n")

## 3. Optional: Build dense embeddings (CPU, ~1-2 min on Colab)

In [ ]:
# Uncomment to enable hybrid mode
# !pip install -q fastembed numpy
# from rag import embed_index
# embed_index(ROOT / "data/index")
# rec_hybrid = PlutusRecommender(ROOT / "data/index", mode="hybrid")
# print(rec_hybrid.recommend("CIP-32 inline datums", k=3))

## 4. Launch a local Gradio demo (optional)

In [ ]:
# !pip install -q gradio
# import app  # will launch the UI
print("To deploy: upload the plutus_recommender folder to a Hugging Face Gradio Space.")